In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_finalproject")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlspchasiquiza01")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/customers_v2.csv"

In [0]:
df_customers = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
schema_customers = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("created_date", StringType(), True)
])

In [0]:
df_customers_final = spark.read\
.option('header', True)\
.schema(schema_customers)\
.csv(ruta)

In [0]:
customers_final_df = df_customers_final.withColumn("ingestion_date", current_timestamp())

In [0]:
customers_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.customers_bz")